# Criando roteador de LLM


In [1]:
import nest_asyncio
nest_asyncio.apply()

In [2]:
from pydantic import BaseModel, Field
class Test(BaseModel):
    respostas: str = Field(description="Respostas geradas pelo modelo.")

In [3]:
from groq import AsyncGroq
from pydantic import BaseModel
import json
import asyncio
from typing import Optional, Dict, Any, Union
import logging

# Configurar logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# client = Groq()
client = AsyncGroq()

class RouterGroq:
    def __init__(self, messages: str, model_llm: str, strutured_output: Optional[BaseModel] = None):
        self.messages = messages
        self.strutured_output = strutured_output
        self.model_llm = model_llm
    
    async def llm_structured_groq(self) -> Dict[str, Any]:
        """
        Chama modelo Groq com saída estruturada
        
        """
        
        if self.strutured_output is None:
                raise ValueError(
                    "structured_output precisa estar definido para usar essa função."
                )
        try:
            response = await client.chat.completions.create(
                model=self.model_llm,
                messages=[
                    {"role": "user", "content": self.messages},  
                ],
                response_format={
                    "type": "json_schema",
                    "json_schema": {
                        "name": "structured_response",
                        "schema": self.strutured_output.model_json_schema(),
                    }
                }
            )
            content = response.choices[0].message.content or "{}"
            return json.loads(content)
        except Exception as e:  # pylint: disable=broad-exception-caught
            logger.error("Erro ao chamar modelo Groq com saída estruturada: %s", e)
            
    
    async def llm_groq(self) -> str:
        """
        Chama modelo Groq sem saída estruturada
        """
        response = await client.chat.completions.create(
            model=self.model_llm,
            messages=[
                {"role": "user", "content": self.messages},  
            ]
        )
        return response.choices[0].message.content or ""
        


import asyncio

groq_router = RouterGroq(
    messages="Explique a teoria da relatividade de forma simples.",
    model_llm="openai/gpt-oss-120b", 
    strutured_output=Test
     
)

In [4]:
groq_router_response = await groq_router.llm_structured_groq()
print(groq_router_response)

INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


{'respostas': 'A teoria da relatividade, desenvolvida por Einstein, tem duas partes: a relatividade restrita e a relatividade geral. A relatividade restrita (1905) afirma que as leis da física são as mesmas para todos os observadores em movimento uniforme e que a velocidade da luz é constante para todos, independentemente do movimento da fonte. Isso leva a efeitos como dilatação do tempo (um relógio em movimento rápido parece andar mais devagar) e contração do comprimento, além da famosa equação E=mc², que mostra que energia e massa são equivalentes. A relatividade geral (1915) descreve a gravidade não como uma força, mas como a curvatura do espaço‑tempo causada por massa e energia; objetos massivos curvam o espaço‑tempo e outros objetos seguem essas curvaturas, o que percebemos como atração gravitacional. Essa teoria explica a órbita dos planetas, a lente gravitacional, a precessão de Mercúrio e a expansão do universo, entre outros fenômenos.'}


In [5]:
import logging
import os
from typing import Optional

from cerebras.cloud.sdk import AsyncCerebras  # pylint: disable=E0401,E0611
from dotenv import load_dotenv
from pydantic import BaseModel

load_dotenv()

logging.basicConfig(level=logging.INFO)

logger = logging.getLogger(__name__)


class RouterCerebras:
    def __init__(
        self,
        messages: str,
        model_llm: str,
        strutured_output: Optional[BaseModel] = None,
    ):
        self.messages = messages
        self.model_llm = model_llm
        self.strutured_output = strutured_output
        self.client_cerebras = self._get_client()

    def _get_client(self) -> AsyncCerebras:
        api_key = os.getenv("CEREBRAS_API_KEY")
        if not api_key:
            raise ValueError(
                "CEREBRAS_API_KEY não está definida nas variáveis de ambiente."
            )
        return AsyncCerebras(api_key=api_key)

    async def get_response_cerebras_structured_async(self) -> Optional[BaseModel]:
        if self.strutured_output is None:
            raise ValueError(
                "structured_output precisa estar definido para usar essa função."
            )
        try:
            response = await self.client_cerebras.chat.completions.create(
                model=self.model_llm,
                messages=[{"role": "user", "content": self.messages}],
                response_format={
                    "type": "json_schema",
                    "json_schema": {
                        "name": "structured_response",
                        "schema": self.strutured_output.model_json_schema(),
                    },
                },
            )
            return response.choices[0].message.content  # type:ignore
        except Exception as e:  # pylint: disable=broad-exception-caught
            logger.error("Erro no llm_pydanticai: %s", e)
            raise

    async def get_response_cerebras_async(self) -> str:
        try:
            response = await self.client_cerebras.chat.completions.create(
                model=self.model_llm,
                messages=[{"role": "user", "content": self.messages}],
            )
            return response.choices[0].message.content  # type:ignore
        except Exception as e:  # pylint: disable=broad-exception-caught
            logger.error("Erro no llm_pydanticai: %s", e)
            raise


In [6]:
router_cerebras = RouterCerebras(
    messages="Explique a teoria da relatividade de forma simples.",
    model_llm="qwen-3-235b-a22b-instruct-2507", 
    strutured_output=Test
)
#await router_cerebras.get_response_cerebras_async()

INFO:httpx:HTTP Request: GET https://api.cerebras.ai/v1/tcp_warming "HTTP/1.1 200 OK"


In [7]:
await router_cerebras.get_response_cerebras_structured_async()

INFO:httpx:HTTP Request: POST https://api.cerebras.ai/v1/chat/completions "HTTP/1.1 200 OK"


'{"respostas": "A teoria da relatividade, criada por Albert Einstein, tem duas partes: a relatividade especial e a relatividade geral. A relatividade especial diz que o tempo e o espaço não são absolutos: eles mudam dependendo da velocidade de quem está observando. Por exemplo, quanto mais rápido você se move, mais devagar o tempo passa para você em comparação com alguém parado. Já a relatividade geral explica a gravidade não como uma força, mas como uma curvatura do espaço e do tempo causada por objetos com massa, como planetas e estrelas. Imagine colocar uma bola pesada em uma rede elástica: ela faz um buraco e puxa outros objetos em volta. É assim que a Terra orbita o Sol — ele \'entorta\' o espaço ao seu redor."}'

In [8]:
import logging
from typing import Any, Optional

from dotenv import load_dotenv
from pydantic import BaseModel
from pydantic_ai import Agent  # pylint: disable=E0401,E0611

# pylint: disable=E0401,E0611 #type:ignore;
from pydantic_ai.models.huggingface import HuggingFaceModel  # pylint: disable=E0401,E0611 #type:ignore;

load_dotenv()

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)


class RouterPydanticAI:
    def __init__(
        self,
        messages: str,
        model_llm: str,
        strutured_output: Optional[BaseModel] = None,
    ):
        self.messages = messages
        self.strutured_output = strutured_output
        self.model_llm = model_llm

    async def llm_structured_pydanticai(self) -> Any:
        """
        Chama modelo HuggingFace via Pydantic AI com saída estruturada
        """
        try:
            if self.strutured_output is None:
                raise ValueError(
                    "structured_output precisa estar definido para usar essa função."
                )

            model = HuggingFaceModel(self.model_llm)
            agent = Agent(model, output_type=self.strutured_output)  # type:ignore

            response = agent.run_sync(self.messages)

            return response

        except Exception as e:  # pylint: disable=broad-exception-caught
            logger.error("Erro no llm_pydantic_structured: %s", e)
            raise

    async def llm_pydanticai(self) -> Any:
        """
        Chama modelo HuggingFace via Pydantic AI sem saída estruturada
        """
        try:
            model = HuggingFaceModel(self.model_llm)
            agent = Agent(model)

            response = agent.run_sync(self.messages)

            return response

        except Exception as e:  # pylint: disable=broad-exception-caught
            logger.error("Erro no llm_pydanticai: %s", e)
            raise

c:\Users\jefer\Documents\Ciencia-de-dados\LLMs\Agente-investimento\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
import asyncio
llm = RouterPydanticAI(
    messages="Explique a teoria da relatividade de forma simples.",
    model_llm="Qwen/Qwen3-235B-A22B-Instruct-2507", 
    #strutured_output=SupervisorResponse
)

#await llm.llm_structured_pydanticai()
await llm.llm_pydanticai()

AgentRunResult(output='Claro! Vamos explicar a **teoria da relatividade** de forma simples, dividindo em duas partes: a **Relatividade Restrita** e a **Relatividade Geral**, ambas criadas por Albert Einstein.\n\n---\n\n### 1. **Relatividade Restrita (1905)**\n\nEssa parte da teoria fala sobre como o **tempo e o espaço** não são absolutos — eles dependem de quem está observando e de quão rápido essa pessoa está se movendo.\n\n#### Ideias principais:\n\n- **Tudo é relativo**: O movimento só faz sentido quando comparado a algo. Por exemplo, se você está sentado em um trem, para você está parado, mas para alguém do lado de fora, você está se movendo.\n\n- **A velocidade da luz é constante**: Não importa quão rápido você vá, a luz sempre se move à mesma velocidade (cerca de 300 mil km/s no vácuo). Isso é estranho, porque se você correr atrás de um feixe de luz, ele ainda vai se afastar de você na mesma velocidade!\n\n- **Tempo e espaço se ajustam**: Para manter a velocidade da luz constante

In [10]:
import logging
from typing import Any, Optional

#from langchain.chat_models import init_chat_model
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from pydantic import BaseModel

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)


class RouterNvidia:
    def __init__(
        self,
        messages: str,
        model_llm: str,
        strutured_output: Optional[BaseModel] = None,
    ):
        self.messages = messages
        self.model_llm = model_llm
        self.strutured_output = strutured_output

    async def llm_nvidia(self) -> Optional[str]:
        """
        Chama modelo Nvidia via LangChain
        """
        try:
            #llm = init_chat_model(self.model_llm, model_provider="nvidia")
            llm = ChatNVIDIA(model=self.model_llm)
            response = await llm.ainvoke([{"role": "user", "content": self.messages}])
            return response.content  # type:ignore

        except Exception as e:  # pylint: disable=broad-exception-caught
            logger.error("Erro no llm_nvidia: %s", e)
            raise

    async def llm_nvidia_structured(self) -> Optional[Any] | BaseModel:
        """
        Chama modelo Nvidia via LangChain com saída estruturada
        """
        if self.strutured_output is None:
            raise ValueError(
                "structured_output precisa estar definido para usar essa função."
            )

        try:
            llm = ChatNVIDIA(model=self.model_llm)
            llm_strutured = llm.with_structured_output( # type:ignore
                self.strutured_output  # type:ignore
            )

            response = await llm_strutured.ainvoke(  # type:ignore
                [{"role": "user", "content": self.messages}]
            )  # type:ignore
            
            if response is None:
                logger.warning(f"Resposta nula recebida do modelo Nvidia. O modelo {self.model_llm} pode não suportar saída estruturada.")
            
            return response  # type:ignore

        except Exception as e:  # pylint: disable=broad-exception-caught
            logger.error("Erro no llm_nvidia_structured: %s", e)
            raise
        
llm_code = RouterNvidia(
    messages="Me explique a teoria da relatividade de forma simples.",
    model_llm="mistralai/mixtral-8x22b-instruct-v0.1",  
    strutured_output=Test
    
)

await llm_code.llm_nvidia_structured()

c:\Users\jefer\Documents\Ciencia-de-dados\LLMs\Agente-investimento\.venv\Lib\site-packages\langchain_nvidia_ai_endpoints\chat_models.py:1066: UserWarning: Model 'mistralai/mixtral-8x22b-instruct-v0.1' is not known to support structured output. Your output may fail at inference time.
  warnings.warn(
c:\Users\jefer\Documents\Ciencia-de-dados\LLMs\Agente-investimento\.venv\Lib\site-packages\pydantic\v1\main.py:1054: UserWarning: LangSmith now uses UUID v7 for run and trace identifiers. This warning appears when passing custom IDs. Please use: from langsmith import uuid7
            id = uuid7()
Future versions will require UUID v7.
  input_data = validator(cls_, input_data)


Test(respostas='1. Comece supondo que exista uma lei física fundamental que é a mesma para todos os observadores, independentemente de sua velocidade ou direção. Isso é chamado de Princípio da Relatividade de Einstein. 2. Em seguida, aceite o fato de que a velocidade da luz no vazio é a mesma para todos os observadores, independentemente de suas velocidades relativas. Isso é chamado de Princípio da Constância da Velocidade da Luz. 3. Suponha que você tenha um relógio e uma longa vara de medir. Você está andando de trem a uma velocidade muito alta. Alguém na plataforma está observando você. De acordo com as teorias de Einstein, suas medidas de tempo e comprimento seriam diferentes. O tempo pareceria se passar mais devagar para a pessoa que estivesse dentro do trem, e a vara pareceria mais pequena para a pessoa na plataforma. Isso é chamado de dilatação do tempo e contração de comprimento. 4. Outro aspecto da Teoria da Relatividade é que massa e energia são equivalentes. Isso significa q

In [11]:
import json
import logging
from typing import Any, Dict, Optional, Union

from pydantic import BaseModel

#from .router_cerebras import RouterCerebras
#from .router_groq import RouterGroq
#from .router_nvidia import RouterNvidia
#from .router_pydantic_ai import RouterPydanticAI

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)


class AllProvidersFailedError(Exception):
    """Exceção levantada quando todos os provedores LLM falham"""


class LlmRouter:
    """
    Classe para roteamento automático entre diferentes modelos LLM com fallback
    """

    def __init__(
        self,
        messages: str,
        strutured_output: Optional[BaseModel] = None,
        groq_models: Optional[list[str]] = None,
        huggingface_models: Optional[list[str]] = None,
        nvidia_models: Optional[list[str]] = None,
        cerebras_models: Optional[list[str]] = None,
    ):
        self.messages = messages
        self.strutured_output = strutured_output
        self.groq_models = groq_models or [
            "moonshotai/kimi-k2-instruct-0905",
            "moonshotai/kimi-k2-instruct",
            "meta-llama/llama-4-scout-17b-16e-instruct",
            "openai/gpt-oss-120b",
        ]

        self.huggingface_models = huggingface_models or [
            "Qwen/Qwen3-235B-A22B-Instruct-2507",
            "microsoft/DialoGPT-large",
            "meta-llama/Llama-4-Maverick-17B-128E",
        ]

        self.nvidia_models = nvidia_models or [
            "microsoft/phi-4-mini-instruct",
            "nvidia/nemotron-4-mini-hindi-4b-instruct",
        ]
        self.cerebras_models = cerebras_models or [
            "qwen-3-235b-a22b-instruct-2507",
            "gpt-oss-120b",
            "llama-4-scout-17b-16e-instruct",
            "qwen-3-32b",
            "llama3.1-8b",
            "llama-4-maverick-17b-128e-instruct",
        ]

    async def try_groq_models(self) -> Optional[Union[Dict[str, Any], str]]:
        """
        Tenta usar modelos Groq em ordem de prioridade
        """
        for model in self.groq_models:
            logger.info("Tentando modelo Groq: %s", model)
            try:
                llm_response = RouterGroq(self.messages, model, self.strutured_output)

                if self.strutured_output:
                    result = await llm_response.llm_structured_groq()
                else:
                    result = await llm_response.llm_groq()

                logger.info("Sucesso com modelo Groq: %s", model)
                return result
            except (ConnectionError, TimeoutError, ValueError) as e:
                logger.warning("Falha no modelo Groq %s: %s", model, e)
                continue
            except Exception as e:  # pylint: disable=broad-except
                logger.warning("Falha no modelo Groq %s: %s", model, e)
                continue

    async def try_huggingface_models(self) -> Any:
        """
        Tenta usar modelos HuggingFace em ordem de prioridade
        """
        for model in self.huggingface_models:
            try:
                llm_response = RouterPydanticAI(
                    self.messages, model, self.strutured_output
                )

                if self.strutured_output:
                    result = await llm_response.llm_structured_pydanticai()
                else:
                    result = await llm_response.llm_pydanticai()

                logger.info("Sucesso com modelo HuggingFace: %s", model)
                return result

            except Exception as e:  # pylint: disable=broad-except
                logger.warning("Falha no modelo Huggingface %s: %s", model, e)
                continue

    async def try_nvidia_models(self) -> Any:
        """
        Tenta usar modelos Nvidia em ordem de prioridade
        """
        for model in self.nvidia_models:
            try:
                llm_response = RouterNvidia(self.messages, model, self.strutured_output)

                if self.strutured_output:
                    result = await llm_response.llm_nvidia_structured()
                else:
                    result = await llm_response.llm_nvidia()

                logger.info("Sucesso com modelo Nvidia: %s", model)
                return result

            except Exception as e:  # pylint: disable=broad-except
                logger.warning("Falha no modelo Nvidia %s: %s", model, e)
                continue

    async def try_cerebras_models(self) -> Any:
        """
        Tenta usar modelos Cerebras em ordem de prioridade
        """
        for model in self.cerebras_models:
            try:
                llm_response = RouterCerebras(
                    self.messages, model, self.strutured_output
                )

                if self.strutured_output:
                    result = await llm_response.get_response_cerebras_structured_async()
                else:
                    result = await llm_response.get_response_cerebras_async()

                logger.info("Sucesso com modelo Cerebras: %s", model)

                return result

            except Exception as e:  # pylint: disable=broad-except
                logger.warning("Falha no modelo Cerebras %s: %s", model, e)
                continue

    async def llm_router(self) -> Union[Dict[str, Any], str, Any]:
        """
        Método principal que implementa o sistema de fallback
        """
        logger.info("Iniciando roteamento LLM")

        try:
            return await self.try_cerebras_models()
        except Exception as cerebras_error:  # pylint: disable=broad-except
            logger.warning("Todos os modelos Cerebras falharam: %s", cerebras_error)
            try:
                return await self.try_nvidia_models()
            except Exception as nvidia_error:  # pylint: disable=broad-except
                logger.warning("Todos os modelos Nvidia falharam: %s", nvidia_error)
                try:
                    return await self.try_groq_models()
                except Exception as groq_error:  # pylint: disable=broad-except
                    logger.warning("Todos os modelos Groq falharam: %s", groq_error)

                    # Segunda tentativa: Modelos HuggingFace
                    try:
                        return await self.try_huggingface_models()
                    except Exception as hf_error:  # pylint: disable=broad-except
                        logger.error(
                            "Todos os modelos HuggingFace falharam: %s", hf_error
                        )
                        raise AllProvidersFailedError(
                            f"Todos os provedores falharam. "
                            f"Groq: {groq_error}, "
                            f"HuggingFace: {hf_error}, "
                            f"Nvidia: {nvidia_error}, "
                            f"Cerebras: {cerebras_error}"
                        ) from cerebras_error

In [12]:
router_structured = LlmRouter(
            messages="Explique a teoria da relatividade de forma simples.",
            
        )
    
response= await router_structured.llm_router()

print(response)

INFO:__main__:Iniciando roteamento LLM
INFO:httpx:HTTP Request: GET https://api.cerebras.ai/v1/tcp_warming "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.cerebras.ai/v1/chat/completions "HTTP/1.1 200 OK"
INFO:__main__:Sucesso com modelo Cerebras: qwen-3-235b-a22b-instruct-2507


Claro! Vamos explicar a **teoria da relatividade** de forma simples e direta. Existem duas partes principais: a **Relatividade Restrita** e a **Relatividade Geral**, ambas criadas por Albert Einstein.

---

### 1. **Relatividade Restrita** (1905)

Imagine que você está em um trem andando muito rápido, sem janelas. Se o trem andar em linha reta e sem sacolejos, você não vai conseguir perceber se está em movimento ou parado — tudo dentro do trem parece normal.

Aqui estão as ideias principais dessa teoria:

- **Tudo é relativo**: Movimento depende do ponto de vista. Por exemplo, se você joga uma bola para cima dentro de um trem em movimento, ela cai na sua mão, como se o trem estivesse parado. Mas para alguém fora do trem, a bola faz uma trajetória curva. Ambos estão certos — depende do referencial.

- **A velocidade da luz é constante**: Não importa se você está parado ou se está se movendo muito rápido, a luz sempre viaja à mesma velocidade no vácuo: cerca de **300 mil km por segundo**

In [13]:
response = await router_structured.try_cerebras_models()

INFO:httpx:HTTP Request: GET https://api.cerebras.ai/v1/tcp_warming "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.cerebras.ai/v1/chat/completions "HTTP/1.1 200 OK"
INFO:__main__:Sucesso com modelo Cerebras: qwen-3-235b-a22b-instruct-2507


In [14]:
response

'Claro! Vamos explicar a **teoria da relatividade** de forma simples. Existem duas partes principais: a **Relatividade Restrita** e a **Relatividade Geral**, ambas criadas por Albert Einstein.\n\n---\n\n### 1. **Relatividade Restrita (1905)**\n\nImagine que você está em um trem andando bem rápido, e dentro dele você acende uma lanterna. Para você, a luz da lanterna vai para frente como sempre. Mas para alguém do lado de fora, parado na estação, o que acontece?\n\nAqui está a ideia principal:\n\n- **A velocidade da luz é sempre a mesma**, não importa se você está parado, correndo ou em um foguete. Ela vale cerca de **300.000 km/s** em todos os casos.\n  \n- Por causa disso, **o tempo e o espaço não são absolutos** — eles mudam dependendo de como você está se movendo.\n\n➡️ **Consequências legais**:\n- **O tempo pode passar mais devagar** para quem está se movendo muito rápido (é o chamado "dilatação do tempo").\n- **Objetos em movimento rápido parecem mais curtos** na direção do movimen

# Tools

## Tool fundamentos

In [15]:
from src.agente_investimento import TratatandoDadosFundamentalistasComparacao

ModuleNotFoundError: No module named 'src'

In [ ]:
import requests
import pandas as pd
from bs4 import BeautifulSoup

def get_ibov_tickers_from_statusinvest():
    url = "https://statusinvest.com.br/indices/ibovespa"
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)",
    }
    r = requests.get(url, headers=headers)
    r.raise_for_status()

    soup = BeautifulSoup(r.text, "html.parser")

    # A página exibe os tickers dentro de elementos com classe .ticker (como você mencionou)
    tickers = []
    for elem in soup.select(".main-list .ticker"):
        t = elem.get_text(strip=True)
        # Alguns podem ter espaço ou texto adicional — limpar
        if t:
            tickers.append(t)
    tickers = list(dict.fromkeys(tickers))  # remove duplicados preservando ordem
    return tickers

In [17]:
tickers = get_ibov_tickers_from_statusinvest()

In [18]:
import warnings
from typing import List

import pandas as pd

warnings.filterwarnings("ignore")


class VerificadorTicks:
    def __init__(self, tic: str):
        self.tic = tic

    def obtendo_ticks(self) -> List[str]:
        list_tic = pd.read_csv(
            "https://raw.githubusercontent.com/Jeferson100/fundamentalist-stock-brazil/main/dados/setor.csv"
        )
        return list_tic["tic"].to_list()

    def verificando_ticks(self) -> bool:
        if self.tic in self.obtendo_ticks():
            return True
        else:
            return False

In [19]:
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)
class DadosFundamentalistas:
    def __init__(
        self,
        tic: str,
    ):
        if VerificadorTicks(tic).verificando_ticks():
            self.tic = tic
        else:
            logger.error(f"O ticker {tic} não existe ou não está disponível em nossa base.")

In [20]:
trat = TratatandoDadosFundamentalistasComparacao(tics=["VALE3"], data_inicio="2022-01-01", data_fim="2024-10-30")

In [21]:
response = await trat.coletando_dados_tickers()

In [22]:
response

[    receita_liquida   ebitda lucro_por_acao       datas    tic  \
 0           56720.0  39110.0           4.61  2022-03-31  VALE3   
 1           54970.0  24840.0           6.01  2022-06-30  VALE3   
 2           52080.0  19310.0           4.87  2022-09-30  VALE3   
 3           62740.0  23450.0           4.09  2022-12-31  VALE3   
 4           43840.0  18080.0           2.10  2023-03-31  VALE3   
 5           47760.0  18060.0           1.01  2023-06-30  VALE3   
 6           51960.0  20310.0           3.05  2023-09-30  VALE3   
 7           64510.0  24120.0           2.64  2023-12-31  VALE3   
 8           41890.0  16270.0           1.83  2024-03-31  VALE3   
 9           51740.0  25100.0           3.21  2024-06-30  VALE3   
 10          52980.0  21360.0           2.95  2024-09-30  VALE3   
 
    alavancagem_financeira  margem_liquida preço_lucro preço_vpa  \
 0                    2.47           0.402        3.63      2.30   
 1                    2.38           0.412        3.44    

In [23]:
fundamentos_markdow = pd.concat(response).to_markdown()

In [24]:
fundamentos_markdow

'|    |   receita_liquida |   ebitda |   lucro_por_acao | datas      | tic   |   alavancagem_financeira |   margem_liquida |   preço_lucro |   preço_vpa |   fluxo_caixa_operacional |   divida_liquida_ebitda |   aumento_reducao_caixa_equivalentes |\n|---:|------------------:|---------:|-----------------:|:-----------|:------|-------------------------:|-----------------:|--------------:|------------:|--------------------------:|------------------------:|-------------------------------------:|\n|  0 |             56720 |    39110 |             4.61 | 2022-03-31 | VALE3 |                     2.47 |            0.402 |          3.63 |        2.3  |                     13780 |                    0.16 |                               -22480 |\n|  1 |             54970 |    24840 |             6.01 | 2022-06-30 | VALE3 |                     2.38 |            0.412 |          3.44 |        1.9  |                     17610 |                    0.24 |                                -5300 |\n|  2 | 

In [25]:
prompt = f"""Com base nos dados fundamentalistas abaixo em formato de tabela markdown, gere uma análise detalhada sobre a saúde financeira e perspectivas de investimento para as empresas listadas.  Identifique tendências positivas ou negativas, pontos fortes e fracos, e forneça recomendações de investimento fundamentadas.
{fundamentos_markdow}
"""

In [26]:
len(prompt)

3522

In [56]:
router_structured = LlmRouter(
            prompt
        )
    
response= await router_structured.llm_router()

print(response)

INFO:__main__:Iniciando roteamento LLM
INFO:httpx:HTTP Request: GET https://api.cerebras.ai/v1/tcp_warming "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.cerebras.ai/v1/chat/completions "HTTP/1.1 200 OK"
INFO:__main__:Sucesso com modelo Cerebras: qwen-3-235b-a22b-instruct-2507
INFO:httpx:HTTP Request: GET https://api.cerebras.ai/v1/tcp_warming "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.cerebras.ai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
ERROR:__main__:Erro no llm_pydanticai: Error code: 429 - {'message': "We're experiencing high traffic right now! Please try again soon.", 'type': 'too_many_requests_error', 'param': 'queue', 'code': 'queue_exceeded'}
INFO:httpx:HTTP Request: GET https://api.cerebras.ai/v1/tcp_warming "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.cerebras.ai/v1/chat/completions "HTTP/1.1 404 Not Found"
ERROR:__main__:Erro no llm_pydanticai: Error code: 404 - {'message': 'Model llama-4-scout-17b-16e-instruct does no

None


In [55]:
response

## Tool metrics

In [25]:
import requests
import pandas as pd
from bs4 import BeautifulSoup

def get_ibov_tickers_from_statusinvest():
    url = "https://statusinvest.com.br/indices/ibovespa"
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)",
    }
    r = requests.get(url, headers=headers)
    r.raise_for_status()

    soup = BeautifulSoup(r.text, "html.parser")

    # A página exibe os tickers dentro de elementos com classe .ticker (como você mencionou)
    tickers = []
    for elem in soup.select(".main-list .ticker"):
        t = elem.get_text(strip=True)
        # Alguns podem ter espaço ou texto adicional — limpar
        if t:
            tickers.append(t + ".SA")  # se quiser no formato Yahoo Finance

    tickers = list(dict.fromkeys(tickers))  # remove duplicados preservando ordem
    return tickers

In [26]:
tickers = get_ibov_tickers_from_statusinvest()

In [27]:
tickers

['VALE3.SA',
 'ITUB4.SA',
 'PETR4.SA',
 'AXIA3.SA',
 'BBDC4.SA',
 'PETR3.SA',
 'SBSP3.SA',
 'BPAC11.SA',
 'B3SA3.SA',
 'ITSA4.SA',
 'WEGE3.SA',
 'EMBJ3.SA',
 'BBAS3.SA',
 'ABEV3.SA',
 'RDOR3.SA',
 'EQTL3.SA',
 'RENT3.SA',
 'ENEV3.SA',
 'SUZB3.SA',
 'PRIO3.SA',
 'RADL3.SA',
 'VBBR3.SA',
 'VIVT3.SA',
 'TOTS3.SA',
 'UGPA3.SA',
 'BBDC3.SA',
 'GGBR4.SA',
 'CPLE5.SA',
 'CMIG4.SA',
 'BBSE3.SA',
 'TIMS3.SA',
 'RAIL3.SA',
 'ENGI11.SA',
 'CSAN3.SA',
 'AXIA6.SA',
 'MOTV3.SA',
 'MBRF3.SA',
 'LREN3.SA',
 'KLBN11.SA',
 'ALOS3.SA',
 'ASAI3.SA',
 'SANB11.SA',
 'EGIE3.SA',
 'ISAE4.SA',
 'SMFT3.SA',
 'TAEE11.SA',
 'CXSE3.SA',
 'MULT3.SA',
 'CMIN3.SA',
 'CPFE3.SA',
 'CYRE3.SA',
 'PSSA3.SA',
 'HYPE3.SA',
 'FLRY3.SA',
 'COGN3.SA',
 'BRAV3.SA',
 'GOAU4.SA',
 'NATU3.SA',
 'DIRR3.SA',
 'CSNA3.SA',
 'IGTI11.SA',
 'HAPV3.SA',
 'CURY3.SA',
 'POMO4.SA',
 'BRAP4.SA',
 'AZZA3.SA',
 'VIVA3.SA',
 'IRBR3.SA',
 'AURE3.SA',
 'YDUQ3.SA',
 'MGLU3.SA',
 'SLCE3.SA',
 'MRVE3.SA',
 'RECV3.SA',
 'BEEF3.SA',
 'USIM5.SA',
 'CEAB

In [28]:
from datetime import datetime

def validate_date(date_str: str, field_name):
    try:
            datetime.strptime(date_str, "%Y-%m-%d")
    except ValueError:
        raise ValueError(
            f"Erro: '{field_name}' deve estar no formato YYYY-MM-DD. "
            f"Valor recebido: {date_str}"
        )

In [29]:
import os
import sys
import pandas as pd
import numpy as np
import yfinance as yf
from datetime import datetime
from typing import List, Optional

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

#from src.utils import validate_date

def fetch_data(tickers: List[str], start: str, end: Optional[str] = None) -> pd.DataFrame:
    if start is None:
        raise ValueError("O parâmetro 'start' é obrigatório.")
    
    validate_date(start, "start")

    if end is None:
        end = datetime.today().strftime("%Y-%m-%d")
    else:
        validate_date(end, "end")

    if start >= end:
        raise ValueError(f"A data inicial ({start}) deve ser anterior à data final ({end.strftime('%Y-%m-%d')}).")
    
    df = yf.download(tickers, start=start, end=end, progress=False)["Close"]

    df = df.dropna(how="all", axis=1)

    if df.empty:
        raise ValueError("Nenhum dado retornado. Verifique os tickers ou intervalo de datas.")
    
    return df

In [30]:
data_precos = fetch_data(tickers, "2022-01-01")

In [31]:
import pandas as pd
from sklearn.covariance import LedoitWolf
import numpy as np
from scipy.optimize import minimize

def markowitz_opt(returns: pd.DataFrame):
    # === Estatísticas ===
    mu = returns.mean()  # retorno médio esperado
    # matriz de covariância com Ledoit–Wolf shrinkage
    lw = LedoitWolf().fit(returns)
    Sigma = lw.covariance_

    n = len(mu)

    # Retorno alvo (por exemplo, retorno médio da amostra de treino)
    target_return = mu.mean()

    # === Funções auxiliares ===
    def portfolio_performance(w, mu, Sigma):
        ret = np.dot(w, mu)
        vol = np.sqrt(np.dot(w.T, np.dot(Sigma, w)))
        return ret, vol

    def objective(w, mu, Sigma, target):
        ret, vol = portfolio_performance(w, mu, Sigma)
        return vol  # minimiza a volatilidade

    # === Restrições ===
    constraints = (
        {'type': 'eq', 'fun': lambda w: np.sum(w) - 1},              # soma = 1
        {'type': 'eq', 'fun': lambda w: np.dot(w, mu) - target_return},  # retorno alvo
    )
    bounds = tuple((0, 1) for _ in range(n))  # sem short

    # === Otimização ===
    w0 = np.repeat(1/n, n)
    opt = minimize(
        objective, w0, args=(mu, Sigma, target_return),
        method='SLSQP', bounds=bounds, constraints=constraints
    )

    weights = opt.x

    # === Aplicação dos pesos ao conjunto de retornos ===
    markowitz_cr = (1 + (weights * returns).sum(axis=1)).cumprod()

    weights_markowitz = pd.DataFrame(
        np.tile(weights, (len(returns), 1)),  # repete o vetor em todas as linhas
        index=returns.index,
        columns=returns.columns
    )

    return weights_markowitz, markowitz_cr

In [32]:
data_precos.drop(columns=["AURE3.SA", "EMBJ3.SA", "MOTV3.SA", "NATU3.SA"], inplace=True, axis=1) 

In [33]:
weights_markowitz, markowitz_cr = markowitz_opt(data_precos.pct_change().dropna())

In [34]:
weights_markowitz

Ticker,ABEV3.SA,ALOS3.SA,ASAI3.SA,AXIA3.SA,AXIA6.SA,AZZA3.SA,B3SA3.SA,BBAS3.SA,BBDC3.SA,BBDC4.SA,...,TOTS3.SA,UGPA3.SA,USIM5.SA,VALE3.SA,VAMO3.SA,VBBR3.SA,VIVA3.SA,VIVT3.SA,WEGE3.SA,YDUQ3.SA
Date,,,,,,,,,,,,,,,,,,,,,
2022-01-04,0.109964,1.047281e-17,1.667829e-18,0.0,8.103150e-19,0.0,6.418055e-20,0.0,0.0,0.0,...,0.0,0.0,0.0,0.061434,2.533434e-20,0.0,0.0,0.043173,0.033151,0.0
2022-01-05,0.109964,1.047281e-17,1.667829e-18,0.0,8.103150e-19,0.0,6.418055e-20,0.0,0.0,0.0,...,0.0,0.0,0.0,0.061434,2.533434e-20,0.0,0.0,0.043173,0.033151,0.0
2022-01-06,0.109964,1.047281e-17,1.667829e-18,0.0,8.103150e-19,0.0,6.418055e-20,0.0,0.0,0.0,...,0.0,0.0,0.0,0.061434,2.533434e-20,0.0,0.0,0.043173,0.033151,0.0
2022-01-07,0.109964,1.047281e-17,1.667829e-18,0.0,8.103150e-19,0.0,6.418055e-20,0.0,0.0,0.0,...,0.0,0.0,0.0,0.061434,2.533434e-20,0.0,0.0,0.043173,0.033151,0.0
2022-01-10,0.109964,1.047281e-17,1.667829e-18,0.0,8.103150e-19,0.0,6.418055e-20,0.0,0.0,0.0,...,0.0,0.0,0.0,0.061434,2.533434e-20,0.0,0.0,0.043173,0.033151,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-11-13,0.109964,1.047281e-17,1.667829e-18,0.0,8.103150e-19,0.0,6.418055e-20,0.0,0.0,0.0,...,0.0,0.0,0.0,0.061434,2.533434e-20,0.0,0.0,0.043173,0.033151,0.0
2025-11-14,0.109964,1.047281e-17,1.667829e-18,0.0,8.103150e-19,0.0,6.418055e-20,0.0,0.0,0.0,...,0.0,0.0,0.0,0.061434,2.533434e-20,0.0,0.0,0.043173,0.033151,0.0
2025-11-17,0.109964,1.047281e-17,1.667829e-18,0.0,8.103150e-19,0.0,6.418055e-20,0.0,0.0,0.0,...,0.0,0.0,0.0,0.061434,2.533434e-20,0.0,0.0,0.043173,0.033151,0.0


In [35]:
markowitz_cr

Date
2022-01-04    1.002054
2022-01-05    0.988634
2022-01-06    0.988897
2022-01-07    0.984964
2022-01-10    0.981427
                ...   
2025-11-13    1.559010
2025-11-14    1.559520
2025-11-17    1.565878
2025-11-18    1.559049
2025-11-19    1.539501
Length: 972, dtype: float64

In [36]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ===============================================================
# MÉTRICAS
# ===============================================================

def daily_returns(prices):
    return prices.pct_change()[1:]

def annualized_volatility(daily_ret, trading_days=252):
    return daily_ret.std() * np.sqrt(trading_days)

def annualized_mean_return(daily_ret, trading_days=252):
    return daily_ret.mean() * trading_days

def cagr(prices):
    days = (prices.index[-1] - prices.index[0]).days
    years = days / 365.25
    return (prices.iloc[-1] / prices.iloc[0])**(1/years) - 1 if years > 0 else np.nan

def max_drawdown(prices):
    cum_max = prices.cummax()
    drawdowns = (prices - cum_max) / cum_max
    return drawdowns.min()

def avg_drawdown(prices):
    dd = (prices / prices.cummax()) - 1
    negatives = dd < 0
    groups = (negatives.astype(int).diff() != 0).cumsum()
    avg_dds = []
    for _, grp in dd.groupby(groups):
        if (grp < 0).any():
            avg_dds.append(grp.min())
    return np.mean(avg_dds) if avg_dds else 0.0

def calmar_ratio(prices):
    ann_ret = cagr(prices)
    mdd = max_drawdown(prices)
    return ann_ret / abs(mdd) if mdd != 0 else np.nan

def sortino_ratio(daily_ret, required_return=0.0, trading_days=252):
    rr_daily = (1 + required_return)**(1/trading_days) - 1
    downside = daily_ret[daily_ret < rr_daily] - rr_daily
    if downside.size == 0:
        return np.nan
    dd = np.sqrt((downside**2).mean()) * np.sqrt(trading_days)
    ann_ret = annualized_mean_return(daily_ret, trading_days)
    return (ann_ret - required_return) / dd if dd != 0 else np.nan

def compute_metrics(price_df):
    metrics = {}
    for col in price_df.columns:
        p = price_df[col].dropna()
        dr = daily_returns(p)
        metrics[col] = {
            "ann_vol": annualized_volatility(dr),
            "ann_mean_ret": annualized_mean_return(dr),
            "cagr": cagr(p),
            "max_drawdown": max_drawdown(p),
            "avg_drawdown": avg_drawdown(p),
            "calmar": calmar_ratio(p),
            "sortino": sortino_ratio(dr)
        }
    return pd.DataFrame(metrics).T

In [37]:
metrics = compute_metrics(data_precos)

In [38]:
len(metrics.to_markdown())

9359

## Todo lists

In [39]:
WRITE_TODOS_DESCRIPTION = """Create and manage structured task lists for tracking progress through complex workflows.

## When to Use
- Multi-step or non-trivial tasks requiring coordination
- When user provides multiple tasks or explicitly requests todo list  
- Avoid for single, trivial actions

## Structure
- Maintain one list containing multiple todo objects (content, status, id)
- Use clear, actionable content descriptions
- Status must be: pending, in_progress, or completed

## Best Practices  
- Only one in_progress task at a time
- Mark completed immediately when task is fully done
- Always send the full updated list when making changes
- Prune irrelevant items to keep list focused

## Progress Updates
- Call TodoWrite again to change task status or edit content
- Reflect real-time progress; don't batch completions  
- If blocked, keep in_progress and add new task describing blocker

## Parameters
- todos: List of TODO items with content and status fields

## Returns
Updates agent state with new todo list."""

In [40]:
"""TODO management tools for task planning and progress tracking.

This module provides tools for creating and managing structured task lists
that enable agents to plan complex workflows and track progress through
multi-step operations.
"""

from langchain_core.messages import ToolMessage
from langchain_core.tools import InjectedToolCallId, tool
from langgraph.prebuilt import InjectedState
from langgraph.types import Command
import operator
from typing import TypedDict, Literal, List, Annotated
from langchain_core.messages import BaseMessage

class Todo(TypedDict):
    """A structured task item for tracking progress through complex workflows.

    Attributes:
        content: Short, specific description of the task
        status: Current state - pending, in_progress, or completed
    """

    content: str
    status: Literal["pending", "in_progress", "completed"]
    

class DeepInvestiment(TypedDict):
    messages: Annotated[List[BaseMessage], operator.add]
    todos: list[Todo]


@tool(description=WRITE_TODOS_DESCRIPTION,parse_docstring=True)
def write_todos(
    todos: list[Todo], tool_call_id: Annotated[str, InjectedToolCallId]
) -> Command:
    """Create or update the agent's TODO list for task planning and tracking.

    Args:
        todos: List of Todo items with content and status
        tool_call_id: Tool call identifier for message response

    Returns:
        Command to update agent state with new TODO list
    """
    return Command(
        update={
            "todos": todos,
            "messages": [
                ToolMessage(f"Updated todo list to {todos}", tool_call_id=tool_call_id)
            ],
        }
    )


@tool(parse_docstring=True)
def read_todos(
    state: Annotated[DeepInvestiment, InjectedState],
    tool_call_id: Annotated[str, InjectedToolCallId],
) -> str:
    """Read the current TODO list from the agent state.

    This tool allows the agent to retrieve and review the current TODO list
    to stay focused on remaining tasks and track progress through complex workflows.

    Args:
        state: Injected agent state containing the current TODO list
        tool_call_id: Injected tool call identifier for message tracking

    Returns:
        Formatted string representation of the current TODO list
    """
    todos = state.get("todos", [])
    if not todos:
        return "No todos currently in the list."

    result = "Current TODO List:\n"
    for i, todo in enumerate(todos, 1):
        status_emoji = {"pending": "⏳", "in_progress": "🔄", "completed": "✅"}
        emoji = status_emoji.get(todo["status"], "❓")
        result += f"{i}. {emoji} {todo['content']} ({todo['status']})\n"

    return result.strip()

In [41]:
TODO_USAGE_INSTRUCTIONS = """Based upon the user's request:
1. Use the write_todos tool to create TODO at the start of a user request, per the tool description.
2. After you accomplish a TODO, use the read_todos to read the TODOs in order to remind yourself of the plan. 
3. Reflect on what you've done and the TODO.
4. Mark you task as completed, and proceed to the next TODO.
5. Continue this process until you have completed all TODOs.

IMPORTANT: Always create a research plan of TODOs and conduct research following the above guidelines for ANY user request.
IMPORTANT: Aim to batch research tasks into a *single TODO* in order to minimize the number of TODOs you have to keep track of.
"""

In [42]:
prompt_agent_investimento = """
<Task>
You are whose mission is to help the user .
Context: today's date is {date}. Assume the user speaks Portuguese unless otherwise specified.

<Capabilities / Tools>
You have access to the following tools:

1. **write_todos** — Manage or update the agent's TODO list for task planning and progress tracking.
   • Limit: up to **3 calls per session** (initial plan + mid-update + final review).

2. **read_todos** — Read the current TODO list to maintain context and focus.
   • Limit: up to **5 calls per session** (use only when switching context or resuming work).

3. **web_search** — Retrieve verified external information (APIs, frameworks, recent updates).
   • Limit: based on complexity level:
     - Simple tasks → **max 1 call**
     - Normal tasks → **max 2 calls**
     - Complex/research-heavy tasks → **max 3 calls**
   • Never exceed 3 total web_search calls in one user task.

4. **think_tool** — Reflect after each search or major step; summarize findings, identify gaps, and plan next actions.
   • Mandatory after every web_search call.
   • Limit: **equal to number of web_search calls + 2** (for initial and final reflections).



<Show Your Thinking>
After each **web_search**, use **think_tool** to summarize:
- Key findings
- Missing information
- Whether another search is needed or if the task can proceed
- How the new info affects the plan



</prompt_agent_code>
"""


In [43]:
from IPython.display import Image, display
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool
from langchain.agents import create_agent
from tavily import TavilyClient
from dotenv import load_dotenv
from langgraph.checkpoint.memory import MemorySaver
import sys
from datetime import datetime


load_dotenv()

tavily_client = TavilyClient()

@tool(parse_docstring=True)
def web_search(query: str, max_results: int = 5, include_images: bool = False) -> list[dict[str, str]]:
    """Search the web for information on a specific topic.

    This tool performs web searches and returns relevant results
    for the given query. Use this when you need to gather information from
    the internet about any topic.

    Args:
        query: The search query string. Be specific and clear about what
               information you're looking for.
        max_results: The maximum number of search results to return.
        include_images: Whether to include images in the search results.

    Returns:
        Search results from search engine.

    Example:
        web_search("machine learning applications in healthcare")
    """
    params = {}
    api_key = os.getenv("TAVILY_API_KEY")
    if not api_key:
        raise ValueError("TAVILY_API_KEY not found in environment variables.")
    params['api_key'] = api_key

    #client = TavilyClient(api_key)

    api_base_url = os.getenv("DLAI_TAVILY_BASE_URL")
    if api_base_url:
        params['api_base_url'] = api_base_url

    client = TavilyClient(api_key=api_key, api_base_url=api_base_url)

    try:
        response = client.search(
            query=query,
            max_results=max_results,
            include_images=include_images
        )

        results = []
        for r in response.get("results", []):
            results.append({
                "title": r.get("title", ""),
                "content": r.get("content", ""),
                "url": r.get("url", "")
            })

        if include_images:
            for img_url in response.get("images", []):
                results.append({"image_url": img_url})

        return results

    except Exception as e:
        return [f'An Error occurred: {e} with the Tavily API. Please try again later.']


@tool(parse_docstring=True)
def think_tool(reflection: str) -> str:
    """Tool for strategic reflection on research progress and decision-making.

    Use this tool after each search to analyze results and plan next steps systematically.
    This creates a deliberate pause in the research workflow for quality decision-making.

    When to use:
    - After receiving search results: What key information did I find?
    - Before deciding next steps: Do I have enough to answer comprehensively?
    - When assessing research gaps: What specific information am I still missing?
    - Before concluding research: Can I provide a complete answer now?
    - How complex is the question: Have I reached the number of search limits?

    Reflection should address:
    1. Analysis of current findings - What concrete information have I gathered?
    2. Gap assessment - What crucial information is still missing?
    3. Quality evaluation - Do I have sufficient evidence/examples for a good answer?
    4. Strategic decision - Should I continue searching or provide my answer?

    Args:
        reflection: Your detailed reflection on research progress, findings, gaps, and next steps

    Returns:
        Confirmation that reflection was recorded for decision-making
    """
    return f"Reflection recorded: {reflection}"


def get_today_str() -> str:
    """Get current date in a human-readable format."""
    if sys.platform.startswith("win"):
        return datetime.now().strftime("%a %b %#d, %Y")
    else:
        return datetime.now().strftime("%a %b %-d, %Y")

model = ChatNVIDIA(
    model="qwen/qwen3-next-80b-a3b-instruct",
)

tools = [write_todos, web_search, read_todos,think_tool]

current_date = get_today_str()
max_concurrent_research_units = 3
max_researcher_iterations = 3

# Create agent
agent = create_agent(
    model,
    tools,
    system_prompt=TODO_USAGE_INSTRUCTIONS
    + "\n\n"
    + "=" * 80
    + "\n\n"
    + "\n\n"
    + prompt_agent_investimento.format(date=get_today_str())
    ,
    state_schema=DeepInvestiment,
    #checkpointer=MemorySaver()
    
)



In [44]:
config = {"configurable": {"user_id": "1", "thread_id": "1"}}

In [47]:
from langchain_core.messages import HumanMessage
rodar = True
if rodar:

    result = agent.invoke(
        {
            "messages": 
                [HumanMessage(role="user",
                            content=""" 
                            Tenho 100000 para investir, monte uma carteira diversificada com ações do Ibovespa.
                                """)],
            
        }, #config=config
    )


In [48]:
from utils import format_messages
if rodar:
    format_messages(result['messages'])

╭─────────────────────────────────────────────────── 🧑 Human ────────────────────────────────────────────────────╮
│                                                                                                                 │
│                             Tenho 100000 para investir, monte uma carteira diversificada com ações do Ibovespa. │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────────────── 📝 AI ─────────────────────────────────────────────────────╮
│                                                                                                                 │
│                                                                                                                 │
│ 🔧 Tool Call: write_todos                                                                                       │
│    Args: {                                                                                                      │
│   "todos": [                                                                                                    │
│     {                                                                                                           │
│       "content": "Pesquisar as principais ações do Ibovespa com bom desempenho histórico e baixa volatilidade", │
│       "status": "pending"                                                                                       │
│     },                                                                                                          │
│     {                                                                                                           │
│       "content": "Identificar setores representativos no Ibovespa para diversificação da carteira",             │
│       "status": "pending"                                                                                       │
│     },                                                                                                          │
│     {                                                                                                           │
│       "content": "Construir uma carteira diversificada com 8-12 ações, distribuindo o capital de R$100.000",    │
│       "status": "pending"                                                                                       │
│     },                                                                                                          │
│     {                                                                                                           │
│       "content": "Revisar e justificar a alocação final da carteira, incluindo riscos e recomendações",         │
│       "status": "pending"                                                                                       │
│     }                                                                                                           │
│   ]                                                                                                             │
│ }                                                                                                               │
│    ID: call_749b17acecdf466ca574927a                                                                            │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 🔧 Tool Output ─────────────────────────────────────────────────╮
│ Updated todo list to [{'content': 'Pesquisar as principais ações do Ibovespa com bom desempenho histórico e     │
│ baixa volatilidade', 'status': 'pending'}, {'content': 'Identificar setores representativos no Ibovespa para    │
│ diversificação da carteira', 'status': 'pending'}, {'content': 'Construir uma carteira diversificada com 8-12   │
│ ações, distribuindo o capital de R$100.000', 'status': 'pending'}, {'content': 'Revisar e justificar a alocação │
│ final da carteira, incluindo riscos e recomendações', 'status': 'pending'}]                                     │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────────────── 📝 AI ─────────────────────────────────────────────────────╮
│                                                                                                                 │
│                                                                                                                 │
│ 🔧 Tool Call: read_todos                                                                                        │
│    Args: {}                                                                                                     │
│    ID: call_0a6e42f840154c8e825aecda                                                                            │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 🔧 Tool Output ─────────────────────────────────────────────────╮
│ Current TODO List:                                                                                              │
│ 1. ⏳ Pesquisar as principais ações do Ibovespa com bom desempenho histórico e baixa volatilidade (pending)     │
│ 2. ⏳ Identificar setores representativos no Ibovespa para diversificação da carteira (pending)                 │
│ 3. ⏳ Construir uma carteira diversificada com 8-12 ações, distribuindo o capital de R$100.000 (pending)        │
│ 4. ⏳ Revisar e justificar a alocação final da carteira, incluindo riscos e recomendações (pending)             │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────────────── 📝 AI ─────────────────────────────────────────────────────╮
│                                                                                                                 │
│                                                                                                                 │
│ 🔧 Tool Call: think_tool                                                                                        │
│    Args: {                                                                                                      │
│   "reflection": "Foi encontrada a composição atualizada do índice IBLV (Ibovespa Smart Low Volatility) da B3,   │
│ com dados de 21/11/2025. As 20 ações que compõem o índice são as de menor volatilidade no Ibovespa, com alta    │
│ liquidez e participação de grandes empresas como Petrobras (PETR4), Itaú Unibanco (ITUB4), Vale (VALE3), Banco  │
│ do Brasil (BBAS3), Eletrobras (ELET6), Cielo (CIEL3) e Bradesco (BBDC4). Além disso, ações com alto dividend    │
│ yield como MRFG3, DIRR3 e CMIN3 também estão presentes. Combinando os dados do IBLV com os dividend yield mais  │
│ altos, é possível construir uma carteira diversificada de 10 ações com baixa volatilidade, sólida fundamentação │
│ e bom retorno por dividendo. Agora, posso prosseguir com a construção da carteira de R$100.000, distribuindo o  │
│ capital entre essas ações com base em liquidez, participação no índice e yield. O próximo passo é definir a     │
│ alocação final, considerando diversificação por setor e limite de exposição individual."                        │
│ }                                                                                                               │
│    ID: call_3bb981572f5f426a99100aa1                                                                            │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 🔧 Tool Output ─────────────────────────────────────────────────╮
│ Reflection recorded: Foi encontrada a composição atualizada do índice IBLV (Ibovespa Smart Low Volatility) da   │
│ B3, com dados de 21/11/2025. As 20 ações que compõem o índice são as de menor volatilidade no Ibovespa, com     │
│ alta liquidez e participação de grandes empresas como Petrobras (PETR4), Itaú Unibanco (ITUB4), Vale (VALE3),   │
│ Banco do Brasil (BBAS3), Eletrobras (ELET6), Cielo (CIEL3) e Bradesco (BBDC4). Além disso, ações com alto       │
│ dividend yield como MRFG3, DIRR3 e CMIN3 também estão presentes. Combinando os dados do IBLV com os dividend    │
│ yield mais altos, é possível construir uma carteira diversificada de 10 ações com baixa volatilidade, sólida    │
│ fundamentação e bom retorno por dividendo. Agora, posso prosseguir com a construção da carteira de R$100.000,   │
│ distribuindo o capital entre essas ações com base em liquidez, participação no índice e yield. O próximo passo  │
│ é definir a alocação final, considerando diversificação por setor e limite de exposição individual.             │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────────────── 📝 AI ─────────────────────────────────────────────────────╮
│                                                                                                                 │
│                                                                                                                 │
│ 🔧 Tool Call: write_todos                                                                                       │
│    Args: {                                                                                                      │
│   "todos": [                                                                                                    │
│     {                                                                                                           │
│       "content": "Pesquisar as principais ações do Ibovespa com bom desempenho histórico e baixa volatilidade", │
│       "status": "completed"                                                                                     │
│     },                                                                                                          │
│     {                                                                                                           │
│       "content": "Identificar setores representativos no Ibovespa para diversificação da carteira",             │
│       "status": "completed"                                                                                     │
│     },                                                                                                          │
│     {                                                                                                           │
│       "content": "Construir uma carteira diversificada com 8-12 ações, distribuindo o capital de R$100.000",    │
│       "status": "in_progress"                                                                                   │
│     },                                                                                                          │
│     {                                                                                                           │
│       "content": "Revisar e justificar a alocação final da carteira, incluindo riscos e recomendações",         │
│       "status": "pending"                                                                                       │
│     }                                                                                                           │
│   ]                                                                                                             │
│ }                                                                                                               │
│    ID: call_c7a72ab5892542c88612e186                                                                            │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 🔧 Tool Output ─────────────────────────────────────────────────╮
│ Updated todo list to [{'content': 'Pesquisar as principais ações do Ibovespa com bom desempenho histórico e     │
│ baixa volatilidade', 'status': 'completed'}, {'content': 'Identificar setores representativos no Ibovespa para  │
│ diversificação da carteira', 'status': 'completed'}, {'content': 'Construir uma carteira diversificada com 8-12 │
│ ações, distribuindo o capital de R$100.000', 'status': 'in_progress'}, {'content': 'Revisar e justificar a      │
│ alocação final da carteira, incluindo riscos e recomendações', 'status': 'pending'}]                            │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────────────── 📝 AI ─────────────────────────────────────────────────────╮
│ Com base nas pesquisas realizadas, aqui está a proposta de **carteira diversificada de ações do Ibovespa** para │
│ um investimento de **R$100.000**, com foco em **baixa volatilidade, alto dividend yield e solidez               │
│ fundamental**, utilizando como base o índice **IBLV (Ibovespa Smart Low Volatility)** e as ações com os         │
│ melhores rendimentos de dividendos em 2025.                                                                     │
│                                                                                                                 │
│ ---                                                                                                             │
│                                                                                                                 │
│ ### 📊 **Carteira Diversificada – R$100.000 (10 ações)**                                                        │
│                                                                                                                 │
│ | Código | Empresa | Setor | Dividend Yield (2025) | Participação na Carteira | Valor Investido | Número de     │
│ Ações (estimado) |                                                                                              │
│ |--------|---------|-------|------------------------|--------------------------|-----------------|------------- │
│ ---------------|                                                                                                │
│ | **PETR4** | Petrobras | Energia | 9,5% | 15% | R$15.000 | ~300 |                                              │
│ | **ITUB4** | Itaú Unibanco | Financeiro | 10,2% | 12% | R$12.000 | ~350 |                                      │
│ | **BBAS3** | Banco do Brasil | Financeiro | 12,0% | 10% | R$10.000 | ~280 |                                    │
│ | **VALE3** | Vale | Mineração | 8,7% | 12% | R$12.000 | ~180 |                                                 │
│ | **ELET6** | Eletrobras | Energia | 7,5% | 8% | R$8.000 | ~400 |                                               │
│ | **CIEL3** | Cielo | Tecnologia | 6,8% | 8% | R$8.000 | ~500 |                                                 │
│ | **MRFG3** | Marfrig | Alimentos | 19,6% | 10% | R$10.000 | ~250 |                                             │
│ | **DIRR3** | Direcional | Logística | 12,3% | 8% | R$8.000 | ~150 |                                            │
│ | **BBDC4** | Bradesco | Financeiro | 11,0% | 8% | R$8.000 | ~300 |                                             │
│ | **CMIN3** | CSN Mineração | Mineração | 14,0% | 9% | R$9.000 | ~200 |                                         │
│                                                                                                                 │
│ **Total:** R$100.000                                                                                            │
│ **Dividend Yield médio da carteira:** ~11,2% ao ano                                                             │
│ **Setores representados:** Financeiro (3), Energia (2), Mineração (2), Alimentos (1), Logística (1), Tecnologia │
│ (1) → **Excelente diversificação**                                                                              │
│                                                                                                                 │
│ ---                                                                                                             │
│                                                                                                                 │
│ ### ✅ **Justificativa da Estratégia**                                                                          │
│                                                                                                                 │
│ - **Baixa volatilidade:** Todas as ações são componentes 